### 🐍 Python Prerequisite: Type Annotations vs. Runtime Behavior

Standard Python includes **type hints**, but Python's runtime **never enforces them**:

```python
def chat(message: str, temperature: float):
    return f"Message: {message}, Temp: {temperature}"

# Python runs this without complaining, even though types are mismatched!
chat(message=123, temperature="high")
```

Pydantic bridges this gap by reading type annotations dynamically at runtime using `pydantic_core` (written in Rust) to enforce types when objects are created.

## 1. Why Pydantic Exists: Parsing & Validation in One Step

When raw external data arrives (e.g., from an HTTP request, JSON file, or LLM payload), it must be both parsed into Python native objects and validated against domain rules.

Pydantic collapses parsing and validation into **one model class, executed automatically upon instantiation**.

> **Resource:** Read more on [Pydantic BaseModel Documentation](https://docs.pydantic.dev/latest/concepts/models/).


In [2]:
from pydantic import BaseModel  #This is the most important line to implement the Pydantic repository in order to implement the validation for type data.

class Product(BaseModel):
    name: str
    price: float
    tags: list[str]
    metadata: dict[str, str]


iphone = Product(name="Iphone", price="12.34", tags=["iphone", 'mobile phone'], metadata={"country": "IN"})

print(iphone)

print(iphone.price)
print(iphone.name)
print(iphone.tags)
print(iphone.metadata)


name='Iphone' price=12.34 tags=['iphone', 'mobile phone'] metadata={'country': 'IN'}
12.34
Iphone
['iphone', 'mobile phone']
{'country': 'IN'}


In [ ]:
from pydantic import BaseModel, ValidationError

class ChatRequest(BaseModel):
    message: str
    temperature: float
    model: str
    dictionary: dict[str, int]

# Automatic type coercion when compatible (e.g., string "0.7" converted to float 0.7)
valid_req = ChatRequest(message="Explain vector databases", temperature=9.05, model="gpt-4o",dictionary={"token":100,"count":1})
print(f"Valid request object created: {valid_req}")
print(f"Temperature type is float: {type(valid_req.temperature)}")

# Automatic validation error on incompatible data types
#This way we can catch the exception
try:
    ChatRequest(message="Explain Vector Database", temperature="0.7", model="gpt-4o",dictionary={"token":100,"count":1})
except ValidationError as e:
    print("\n--- Caught expected Pydantic ValidationError ---")
    print(e)

Valid request object created: message='Explain vector databases' temperature=9.05 model='gpt-4o' dictionary={'token': 100, 'count': 1}
Temperature type is float: <class 'float'>


 ## 3. Required vs. Optional Fields

Pydantic draws a clear boundary between **Allowed Values** and **Requirement Status**:

1. `float | None` defines **allowed types** (it allows numbers or `None`).
2. `= None` (or any default assignment) specifies **if the caller can omit the key**.

> 🔑 **Rule:** Defining a field as `float | None` *without* a default value means it is **still required**—the caller must explicitly provide `None`.

In [ ]:
class ChatRequestOptional(BaseModel):
    message: str                       # Required field
    temperature: float | None = None   # Optional field, defaults to None
    max_tokens: int = 1000             # Optional field, defaults to 1000
    system_prompt: str | None          # Explicitly required field that accepts None!

# Passing required fields
req1 = ChatRequestOptional(message="Hello", system_prompt=None)
print(f"Req 1: {req1}")

# Failing because 'system_prompt' was omitted even though it accepts None
try:
    ChatRequestOptional(message="Hello",system_prompt="Prompt Is here")  #This will pass as all the parameter is provided.
    #ChatRequestOptional(message="Hello")   #This will fail as most of the parameter are not there. 
except ValidationError as e:
    print("\n--- Caught error when missing required field 'system_prompt' ---")
    print(e)


Req 1: message='Hello' temperature=None max_tokens=1000 system_prompt=None


 ## Nested Chats
 

In [10]:
class ModelConfig(BaseModel):
    model: str
    temperature: float

class Message(BaseModel):
    role: str
    content: str

class NestedChatRequest(BaseModel):
    config: ModelConfig
    messages: list[Message]

raw_payload = {
    "config": {"model": "gpt-4o", "temperature": 0.7},
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is RAG?"}
    ]
}

req = NestedChatRequest(**raw_payload)
print(req.config)

model='gpt-4o' temperature=0.7


## 5. Constraint Validation & Aliases with `Field()`

The `Field()` function defines constraints, default values, descriptions, and custom input/output field names (aliases).

| Constraint Parameter | Description | Example |
| :--- | :--- | :--- |
| `min_length` / `max_length` | Restricts string length | `Field(min_length=1, max_length=100)`[cite: 1] |
| `gt` / `ge` | Greater than / Greater than or equal to | `Field(ge=0.0)`[cite: 1] |
| `lt` / `le` | Less than / Less than or equal to | `Field(le=2.0)`[cite: 1] |
| `alias` | Maps external JSON keys to Pythonic field names | `Field(alias="api_key")`[cite: 1] |

In [ ]:
from pydantic import Field

class ConstrainedChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=5000)
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    # Alias maps incoming key 'user_id' to internal variable name 'user_identifier'
    user_identifier: str = Field(alias="user_id")

# Passing payload using the alias name 'user_id', alias will work only, when we are trying to create objects
req = ConstrainedChatRequest(message="Hello", user_id="usr_12345", temperature=1.4)
print(f"Parsed with Alias -> Internal Attribute: {req}")


Parsed with Alias -> Internal Attribute: message='Hello' temperature=1.4 user_identifier='usr_12345'


### 5.1 Modern Idiomatic Standard: Using `typing.Annotated`

While assigning `field_name: type = Field(...)` works, modern Pydantic (v2+) strongly recommends using **`typing.Annotated`** ([PEP 593](https://peps.python.org/pep-0593/)):

```python
# Modern & Preferred Syntax
message: Annotated[str, Field(min_length=1, max_length=5000)] = "Default prompt"

To pass some metadata information along with data. 

In [ ]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

class ConstrainedChatRequest(BaseModel):
    message: Annotated[str, Field(min_length=1, max_length=5000)]
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)

In [13]:
from typing import Annotated
from pydantic import BaseModel, Field, ValidationError

# ------------------------------------------------------------------
# Reusable Custom Domain Types with Annotated
# ------------------------------------------------------------------
PromptText = Annotated[
    str, 
    Field(min_length=1, max_length=5000, description="The primary user prompt")
]
Temperature = Annotated[
    float, 
    Field(ge=0.0, le=2.0, description="Sampling temperature")
]

# ------------------------------------------------------------------
# Model Definition using Modern Annotated Syntax
# ------------------------------------------------------------------
class ConstrainedChatRequest(BaseModel):
    message: PromptText
    temperature: Temperature = 0.7
    
    # Inline Annotated field with alias mapping
    user_identifier: Annotated[str, Field(alias="user_id")]

req = ConstrainedChatRequest(message="Explain Vector DBs", user_id="usr_12345")
print(f"Parsed with Alias -> Internal Attribute: {req.user_identifier}")
print(f"Default Temperature Applied: {req.temperature}")

Parsed with Alias -> Internal Attribute: usr_12345
Default Temperature Applied: 0.7


In [ ]:
from pydantic import field_validator, model_validator

class SignupRequest(BaseModel):
    username: str
    password: str
    confirm_password: str

    # Single Field Transformation / Validation
    @field_validator("username")
    @classmethod
    def validate_username(cls, value: str) -> str:
        if " " in value:
            raise ValueError("Username cannot contain spaces")
        return value.lower()  # Normalizes username to lowercase

    # Cross-Field Validation
    @model_validator(mode="after")
    def verify_passwords_match(self):
        if self.password != self.confirm_password:
            raise ValueError("Passwords do not match")
        return self


signup = SignupRequest(username="ANUJ_KUMar", password="1234", confirm_password="1234")

print(signup)


## 7. Specialized Types: `EmailStr`, `SecretStr`, and `ConfigDict`

Pydantic provides rich specialized types out of the box:

* `EmailStr`: Validates email syntax automatically. *(Note: requires `pip install pydantic[email]`)*
* `SecretStr`: Hides API keys or passwords when printing models to avoid logging sensitive data.
* `ConfigDict(extra="forbid")`: Configuration dictionary to reject unexpected input fields.

> **Resource:** Guide on [Pydantic Settings & Special Types](https://docs.pydantic.dev/latest/concepts/types/).

In [ ]:
from pydantic import ConfigDict, SecretStr, EmailStr

class APIConfiguration(BaseModel):
    # Strict configuration: throw error if extra unexpected keys are sent
    model_config = ConfigDict(extra="forbid")
    
    admin_email: EmailStr
    api_key: SecretStr

config = APIConfiguration(
    admin_email="admin@ai-service.com", 
    api_key="sk-proj-secret-token-12345"
)

print(config)


## 8. Serialization & Deserialization Pipeline

Pydantic standardizes data transformations across Python objects, dictionaries, and JSON strings[cite: 1]:

```
Dict / DB Row ──────> model_validate() ──────> Pydantic Model Instance
Pydantic Model ─────> model_dump() ──────────> Python Dictionary
Pydantic Model ─────> model_dump_json() ─────> JSON String
```

> **Resource:** See [Pydantic Serialization Docs](https://docs.pydantic.dev/latest/concepts/serialization/).

In [20]:
from pydantic import BaseModel, EmailStr

raw_dict = {"name": "Anuj", "age": 28, "email": "anuj@example.com"}

class User(BaseModel):
    name: str
    age: int
    email: EmailStr

# user = User(**raw_dict)
user = User.model_validate(raw_dict)

user_dict = user.model_dump()

user_json = user.model_dump_json()


print(user)
print(type(user_dict))
print(user_json)

name='Anuj' age=28 email='anuj@example.com'
<class 'dict'>
{"name":"Anuj","age":28,"email":"anuj@example.com"}


---

# 📝 Pydantic Homework

Complete the five exercises below by implementing the requested Pydantic models, creating sample instances, and running the validation/serialization checks.


## Homework 1: User Profile Model

Create a Pydantic `User` model with:

- `name`
- `age`
- `email`
- Optional `phone`
- `password` using `SecretStr`

Create an instance and practice direct attribute access.
Also serialize the model and deserialize it back into a Pydantic object.


In [ ]:
from typing import Annotated, Optional
from pydantic import BaseModel, EmailStr, SecretStr, Field

class User(BaseModel):
    name: str
    age: Annotated[int, Field(ge=0, le=120)]
    email: EmailStr
    phone: Optional[str] = None
    password: SecretStr

# Create instance
user = User(
    name="Alice Sharma",
    age=28,
    email="alice@example.com",
    phone="+91-98765-43210",
    password="super-secret-password"
)

# Direct attribute access
print("Name:", user.name)
print("Age:", user.age)
print("Email:", user.email)
print("Phone:", user.phone)
print("Password:", user.password)          # Prints as SecretStr('**********')
print("Password value:", user.password.get_secret_value())

# Serialize to dict (secrets are hidden by default; use a custom serializer if you need them)
user_dict = user.model_dump()
print("\nSerialized dict:", user_dict)

# Deserialize back to Pydantic object
user_again = User.model_validate(user_dict)
print("\nDeserialized object:", user_again)


## Homework 2: Nested Order Model

Create Pydantic models for:

- `Customer`
- `Address`
- `Product`
- `Order`

An `Order` should contain a nested `Customer`, an `Address`, and a collection of `Product` objects.
Validate the input and access nested fields using normal attribute access.


In [ ]:
from typing import Annotated
from pydantic import BaseModel, EmailStr, Field

class Address(BaseModel):
    street: str
    city: str
    zip_code: Annotated[str, Field(min_length=5, max_length=10)]
    country: str = "India"

class Customer(BaseModel):
    id: int
    name: str
    email: EmailStr

class Product(BaseModel):
    id: int
    name: str
    price: Annotated[float, Field(gt=0)]
    quantity: Annotated[int, Field(gt=0)]

class Order(BaseModel):
    order_id: Annotated[str, Field(min_length=1)]
    customer: Customer
    shipping_address: Address
    products: list[Product]

    @property
    def total_amount(self) -> float:
        return sum(p.price * p.quantity for p in self.products)


raw_order = {
    "order_id": "ORD-2026-001",
    "customer": {
        "id": 101,
        "name": "Ravi Kumar",
        "email": "ravi@example.com"
    },
    "shipping_address": {
        "street": "42, MG Road",
        "city": "Bangalore",
        "zip_code": "560001",
        "country": "India"
    },
    "products": [
        {"id": 1, "name": "Wireless Mouse", "price": 999.0, "quantity": 2},
        {"id": 2, "name": "USB-C Cable", "price": 299.0, "quantity": 3}
    ]
}

order = Order.model_validate(raw_order)

# Attribute access on nested fields
print("Order ID:", order.order_id)
print("Customer:", order.customer.name, "<", order.customer.email, ">")
print("City:", order.shipping_address.city)
print("First Product:", order.products[0].name)
print("Total Amount: ₹", order.total_amount)

print("\nSerialized JSON:")
print(order.model_dump_json(indent=2))


## Homework 3: Validated API Request

Create a `UserRequest` model with:

- Required and optional fields
- An aliased field using `Field`
- Constraints such as minimum/maximum length and numeric limits
- `Annotated` for at least one field

Add a custom validator for one business rule.


In [ ]:
from typing import Annotated, Optional
from pydantic import BaseModel, Field, field_validator, ValidationError

Username = Annotated[
    str,
    Field(min_length=3, max_length=20, description="User's public handle")
]

class UserRequest(BaseModel):
    username: Username
    age: Annotated[int, Field(ge=13, le=100)]
    user_id: Annotated[str, Field(alias="user_id")]   # aliased field
    bio: Optional[Annotated[str, Field(max_length=160)]] = None
    country_code: Optional[Annotated[str, Field(min_length=2, max_length=2)]] = "IN"

    @field_validator("username")
    @classmethod
    def no_spaces_and_lowercase(cls, value: str) -> str:
        if " " in value:
            raise ValueError("username must not contain spaces")
        return value.lower()

    @field_validator("country_code")
    @classmethod
    def uppercase_country(cls, value: Optional[str]) -> Optional[str]:
        return value.upper() if value else value


# Valid request using alias
valid = UserRequest(username="Alice_Sharma", age=28, user_id="usr_123", bio="AI enthusiast")
print("Valid:", valid)

# Invalid request: too short username + invalid age
invalid_payload = {
    "username": "al",
    "age": 10,
    "user_id": "usr_999",
    "country_code": "ind"
}

try:
    UserRequest.model_validate(invalid_payload)
except ValidationError as e:
    print("\n--- Validation Errors ---")
    print(e)


## Homework 4: Application Configuration

Create a Pydantic model representing application configuration.

Include:

- API key using `SecretStr`
- Environment/name fields
- A dictionary of model configuration
- Appropriate field constraints
- At least one optional field

Serialize the configuration and verify that sensitive values are handled appropriately.


In [ ]:
from typing import Annotated, Optional
from pydantic import BaseModel, SecretStr, Field, ConfigDict, field_validator
import re

class AppConfig(BaseModel):
    model_config = ConfigDict(str_to_lower=True)

    app_name: Annotated[str, Field(min_length=1, max_length=50)]
    environment: str
    api_key: SecretStr
    model_config_map: dict[str, dict[str, float | int | str]]
    max_requests_per_minute: Annotated[int, Field(ge=1, le=10_000)] = 100
    log_level: Optional[Annotated[str, Field(pattern=r"^(debug|info|warning|error)$")]] = "info"

    @field_validator("environment")
    @classmethod
    def validate_environment(cls, value: str) -> str:
        value = value.lower()
        if value not in {"dev", "staging", "prod"}:
            raise ValueError("environment must be one of: dev, staging, prod")
        return value


config = AppConfig(
    app_name="AI Chat Service",
    environment="PROD",                         # Normalized to "prod" by validator
    api_key="sk-live-1234567890abcdef",
    model_config_map={
        "gpt-4o": {"temperature": 0.7, "max_tokens": 2000},
        "gpt-4o-mini": {"temperature": 0.5, "max_tokens": 1000}
    },
    max_requests_per_minute=500,
    log_level="debug"
)

print("Config object:", config)

# Default serialization hides SecretStr
serialized = config.model_dump()
print("\nSerialized dict (safe mode):")
print(serialized)

# Access secret value directly
print("\nAPI key value:", config.api_key.get_secret_value())

# JSON output also hides secrets by default
print("\nJSON output:")
print(config.model_dump_json(indent=2))


## Homework 5: LLM Response Pipeline

Create a Pydantic model representing a structured LLM response:

- `answer`
- `confidence`
- `sources`
- `metadata`

Add appropriate validation constraints and one custom validation rule.
Simulate the complete pipeline:

```text
JSON Response → Pydantic Validation → Python Object → Application Logic → Serialization → JSON
```


In [ ]:
from typing import Annotated, Optional, Any
from pydantic import BaseModel, Field, model_validator, ValidationError

class Source(BaseModel):
    title: str
    url: Annotated[str, Field(pattern=r"^https?://")]

class LLMResponse(BaseModel):
    answer: Annotated[str, Field(min_length=1, max_length=2000)]
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    sources: list[Source]
    metadata: dict[str, Any]

    @model_validator(mode="after")
    def high_confidence_needs_sources(self):
        if self.confidence > 0.9 and len(self.sources) == 0:
            raise ValueError("confidence > 0.9 requires at least one source")
        return self


# 1. Incoming JSON response (as dict)
raw_llm_json = {
    "answer": "Pydantic is a data validation library for Python.",
    "confidence": 0.96,
    "sources": [
        {"title": "Pydantic Docs", "url": "https://docs.pydantic.dev"}
    ],
    "metadata": {
        "model": "gpt-4o",
        "tokens_used": 42,
        "latency_ms": 120
    }
}

# 2. Validate JSON into Pydantic object
response = LLMResponse.model_validate(raw_llm_json)

# 3. Application logic
if response.confidence >= 0.9:
    trust_label = "HIGH"
elif response.confidence >= 0.7:
    trust_label = "MEDIUM"
else:
    trust_label = "LOW"

print(f"Answer: {response.answer}")
print(f"Confidence: {response.confidence} ({trust_label})")
print(f"Sources: {[s.title for s in response.sources]}")
print(f"Model used: {response.metadata.get('model')}")

# 4. Serialize back to dict / JSON
output_dict = response.model_dump()
output_json = response.model_dump_json(indent=2)

print("\n--- Serialized Dict ---")
print(output_dict)

print("\n--- Serialized JSON ---")
print(output_json)

# 5. Invalid case: high confidence without sources
bad_json = {
    "answer": "Wrong answer",
    "confidence": 0.95,
    "sources": [],
    "metadata": {}
}

try:
    LLMResponse.model_validate(bad_json)
except ValidationError as e:
    print("\n--- Caught expected error ---")
    print(e)
